# ICT Backtest + P39 — Colab runner (NO Google Drive login)

This version pulls your data by **public link** — so there is **no Drive sign-in popup**
(that's the step that kept failing on mobile). Nothing to paste, no token.

### Setup — make 3 things public (each is ~3 taps):
1. **The repo:** github.com/ThabisoCollinSengane/Ict → Settings → Danger Zone → **Make public**
2. **Your M1 data folder** ("Backtesting data dxy, gbpusd, eurusd, euruss"): in Drive, ⋮ → Share →
   General access → **Anyone with the link** (Viewer)
3. **Your tick folder** ("Tick data, fx eurusd and gbpusd 2022 and 2024"): same — **Anyone with the link**

Then **Runtime ▸ Run all** and just wait. The backtest result appears first, then P39.
Set the repo + folders back to private/restricted after it finishes.


### 1. Get the code + tools (repo must be Public)


In [ ]:
import os, subprocess
if os.path.isdir('/content/Ict'): subprocess.run(['rm','-rf','/content/Ict'])
r = subprocess.run(['git','clone','--branch','p39-volume-analysis','--depth','1',
                    'https://github.com/ThabisoCollinSengane/Ict.git','/content/Ict'],
                   capture_output=True, text=True)
print(r.stderr[-500:] or 'cloned')
assert os.path.isdir('/content/Ict/scripts'), 'Clone failed — is the repo set to Public?'
os.chdir('/content/Ict')
subprocess.run(['pip','install','-q','-r','requirements.txt','gdown'])
print('code + dependencies ready')


### 2. Download the M1 data (public link — no login)


In [ ]:
import subprocess, glob, os
subprocess.run(['gdown','--folder','--remaining-ok','-O','/content/m1dl',
                'https://drive.google.com/drive/folders/1uN2c7QvNJg15CmVmNXUYR1CTiSsaB-d4'])
ann = glob.glob('/content/m1dl/**/HISTDATA_*_M1????.zip', recursive=True)
assert ann, 'No M1 zips — is the M1 folder set to Anyone-with-the-link?'
M1_DIR = os.path.dirname(ann[0])
print('M1 zips:', len(glob.glob(f'{M1_DIR}/HISTDATA_*_M1*.zip')), 'in', M1_DIR)


### 3. Prepare M1 + run the full backtest (2022–2025)
The backtest summary prints at the end of this cell — that's your R429M / PF 4.47 / MaxDD number.


In [ ]:
import subprocess
subprocess.run(["python","scripts/prepare_histdata.py",M1_DIR])
res = subprocess.run(['python','run_backtest_histdata.py','--years','2022','2023','2024','2025'],
                     capture_output=True, text=True)
open('/content/backtest_output.txt','w').write(res.stdout)
print(res.stdout[-9000:])
if res.returncode: print('--- STDERR ---', res.stderr[-3000:])


### 4. Download the tick data (slow — a few GB by public link)


In [ ]:
import subprocess, glob, os
subprocess.run(['gdown','--folder','--remaining-ok','-O','/content/tickdl',
                'https://drive.google.com/drive/folders/1cXPxh_PqcNYIhHOnvZoV6JRI526tQIz-'])
tz = glob.glob('/content/tickdl/**/HISTDATA_*_T*.zip', recursive=True)
assert tz, 'No tick zips — is the tick folder set to Anyone-with-the-link?'
TICK_DIR = os.path.dirname(tz[0])
print('tick zips:', len(glob.glob(f'{TICK_DIR}/HISTDATA_*_T*.zip')), 'in', TICK_DIR)


### 5. P39 — aggregate ticks + analyse (several minutes)


In [ ]:
import subprocess
subprocess.run(["python","scripts/p39_volume_analysis.py","aggregate",TICK_DIR])
subprocess.run(["python","scripts/p39_volume_analysis.py","analyse"])
print('P39 done')


### 6. Print the results (copy this to Claude)


In [ ]:
import os
print('='*60, '\n  BACKTEST SUMMARY\n', '='*60)
bt = open('/content/backtest_output.txt').read() if os.path.exists('/content/backtest_output.txt') else ''
print(bt[-4000:] or '(no backtest output)')
print('\n'+'='*60, '\n  P39 REPORT\n', '='*60)
p = 'data/p39_volume_report.md'
print(open(p).read() if os.path.exists(p) else '(no P39 report)')


---
**Done.** Copy the two blocks above into the Claude chat (or say *"results are ready"*).
Then set the repo + both folders back to private.
